# Script for testing regression version of CNN 

## Imports


In [1]:
import mne
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold,GroupKFold
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score
import tensorflow as tf
from scipy.signal import resample_poly
from math import gcd
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras import backend as K
import gc

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
mne.set_log_level("CRITICAL")

## Loading in Data

In [2]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all_temp = features_all
features_all = features_all[~((features_all["Subject"] == "RL03JG") & (features_all["Nap Number"] == 5))] # presentation for this subject is 0 ? somethign weird 

labels = pd.read_pickle("segments.pkl")

print("==========SANITY CHECK==========")
print("Nap IDs align? ", (labels["Nap_ID"].values == features_all["Nap Number"].values).all())
print("Subject rows align? ", (labels["Subject"].values == features_all["Subject"].values).all())

features_all = features_all.reset_index(drop=True)
labels = labels.reset_index(drop=True)

features_all = pd.concat([features_all, labels], axis=1)  

==========SANITY CHECK==========
Nap IDs align?  True
Subject rows align?  True


## Helper Functions 

In [3]:
# function for resampling time series data to sampling frequency of epochs 
def resample_timeseries(X, orig_fs, target_fs):

    g = gcd(orig_fs, target_fs)
    up   = target_fs // g   # upsample factor
    down = orig_fs   // g   # downsample factor

    resampled = resample_poly(X, up, down, axis=-1)

    # fix weirdness caused by resampling 
    resampled = (resampled >= 0.5).astype(int)[0:len(resampled)] 
    
    return resampled

In [ ]:
# create custom loss function 
def distillation_loss(teacher_logits, temperature=3.0, alpha=0.5):
    # teacher_logits is captured here in the closure
    def loss_fn(y_true, y_pred):  # keras passes these automatically
        hard_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        soft_pred_new = tf.nn.softmax(y_pred / temperature)
        soft_pred_teacher = tf.nn.softmax(tf.cast(teacher_logits, tf.float32) / temperature)
        soft_loss = tf.keras.losses.KLDivergence()(soft_pred_teacher, soft_pred_new)
        return alpha * soft_loss * (temperature**2) + (1 - alpha) * hard_loss
    return loss_fn  # returns the function keras will call


In [ ]:
def run_kfold_training(model_func, X, y, groups_train_full, input_shape, num_classes, feature_num,compile_kwargs,teacher_preds, fit_kwargs=None, # arguments fed to model.fit() 
                       n_splits=5,random_state=42, shuffle=True, verbose=1,epoch_num=10,
                         type=1,): # model type (1: single head count, 2: two head count and duration, 3: two head for zygo and corr, 4: four head) 
   # returns a dictionary with {"models", "histories", "cv_scores"}.     

    if fit_kwargs is None:
        fit_kwargs = {}
    fit_kwargs = fit_kwargs.copy()

 
    kf = GroupKFold(n_splits=n_splits, random_state=random_state, shuffle=shuffle)

    models = []
    histories = []
    cv_scores = []

    fold = 1
    for train_idx, val_idx in kf.split(X, y, groups=groups_train_full):
        print(f"Fold: {fold} {'='*65}")
        X_train, X_val = X[train_idx], X[val_idx]  # use X_train_full, not X
        y_train, y_val = y[train_idx], y[val_idx]

        model = model_func(input_shape, num_classes, feature_num)

        if type == 1:
            loss_fn = compile_kwargs.get("loss", "sparse_categorical_crossentropy")
        elif type == 2:
            teacher_logits = teacher_preds[train_idx]
            loss_fn = distillation_loss(teacher_logits, temperature=3.0, alpha=0.8)

        
        model.compile(
                optimizer=compile_kwargs["optimizer"],
                loss=loss_fn,
                metrics=compile_kwargs["metrics"]
            )

 
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epoch_num,
        )

        scores = model.evaluate(X_val, y_val, return_dict=True)

        fold += 1
        models.append(model)
        histories.append(history)
        cv_scores.append(scores)

    return {
        "models": models,
        "histories": histories,
        "cv_scores": cv_scores,
    }

## CNN Model

In [ ]:
# single head CNN model for number of contractions  
def CNN_model_contraction(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=3))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?
    model.add(MaxPooling1D(pool_size=1)) 

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='linear'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [ ]:
# single head CNN model for number of contractions  
def CNN_model_contraction_events(input_shape, num_classes,feature_num):

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=3))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?
    model.add(MaxPooling1D(pool_size=1)) 

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

## Defining Training Data 

### Initializing Variables Needed

In [6]:
# global variables for training 
epoch_num = 10
patience = 6
early_stop = EarlyStopping(monitor='val_loss',  patience=patience, restore_best_weights=True)


metrics = ["f1", "accuracy", "mse"]
targets = ["zygo", "corr", "overall"]
splits = ["training", "testing"]
metric_tuples = [("model_name", "", "")]
metric_tuples += [
    (metric, target, split)
    for metric in metrics
    for target in targets
    for split in splits
]
metric_tuples += [("k-fold", "mean", ""), ("k-fold", "std", "")]
results_columns = pd.MultiIndex.from_tuples(
    metric_tuples,
    names=["metric", "target", "split"]
)
results_columns = pd.MultiIndex.from_tuples(metric_tuples)
results_df = pd.DataFrame(columns=results_columns)

### Training Data for Loss CNN

In [8]:
muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_zygo = features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()


indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

subjects_zygo = features_all_temp["Subject"].to_numpy()  # adjust column name
subjects_corr = features_all_temp["Subject"].to_numpy()
groups = np.concatenate([subjects_zygo, subjects_corr])  # mirrors how you built X and y

# Apply same train/test split to groups
groups_train_full = groups[idx_train]
groups_test = groups[idx_test]


# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

### Training Data for Regression CNN

## Training Model

In [ ]:
# call function for training
results_single_head_contraction = run_kfold_training(
    model_func=CNN_model_contraction_events,
    X=X_train_full,
    y=y_train_full,
    groups_train_full=groups_train_full,
    teacher_preds=None,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "metrics": ["accuracy"]
    },
    fit_kwargs={"epochs": 10},
    n_splits=5,
    type=1,)


model_contraction = results_single_head_contraction["models"][-1]
model_history_contraction = model_contraction.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

Fold: 1 =================================================================
Epoch 1/10
286/286 [==============================] - 11s 35ms/step - loss: nan - accuracy: 0.7783 - val_loss: nan - val_accuracy: 0.7228
Epoch 2/10
286/286 [==============================] - 14s 50ms/step - loss: nan - accuracy: 0.7354 - val_loss: nan - val_accuracy: 0.7228
Epoch 3/10
286/286 [==============================] - 10s 36ms/step - loss: nan - accuracy: 0.7354 - val_loss: nan - val_accuracy: 0.7228
Epoch 4/10
286/286 [==============================] - 12s 41ms/step - loss: nan - accuracy: 0.7354 - val_loss: nan - val_accuracy: 0.7228
Epoch 5/10
286/286 [==============================] - 10s 36ms/step - loss: nan - accuracy: 0.7354 - val_loss: nan - val_accuracy: 0.7228
Epoch 6/10
286/286 [==============================] - 11s 39ms/step - loss: nan - accuracy: 0.7354 - val_loss: nan - val_accuracy: 0.7228
Epoch 7/10
286/286 [==============================] - 14s 49ms/step - loss: nan - accuracy: 0.7354

In [ ]:

# call function for training
results_single_head_regression = run_kfold_training(
    model_func=CNN_model_contraction_events,
    X=X_train_full,
    y=y_train_full,
    teacher_preds=model_contraction.predict(X_train_full),
    groups_train_full=groups_train_full,
    input_shape=input_shape,
    num_classes=num_classes,
    feature_num=feature_num,
    compile_kwargs={
        "optimizer": "adam",
        "loss": "sparse_categorical_crossentropy",
        "metrics": ["accuracy"]
    },
    fit_kwargs={"epochs": 10},
    n_splits=5,
    type=2)


model_regression_final = results_single_head_regression["models"][-1]
model_history_regression = model_regression_final.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

### Results

In [ ]:
# full training results (test data not seen during cross val)
y_pred_train = model_contraction.predict(X_train_full)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model_contraction.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   

# Calculate accuracy
accuracy_training = accuracy_score(y_train_full, y_pred_train)   
accuracy_test = accuracy_score(y_test, y_pred_test)  

# Calculate F1 score
f1_training = f1_score(y_train_full, y_pred_train, average='weighted')  
f1_test = f1_score(y_test, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Model scores---------------")
print("Training Accuracy :", accuracy_training)  
print("Test Accuracy :", accuracy_test)  
print("Training F1 Score :", f1_training)   
print("Test F1 Score :", f1_test) 

90/90 [==============================] - 1s 10ms/step
Model scores---------------
Training Accuracy : 0.8928819444444445
Test Accuracy : 0.8545138888888889
Training F1 Score : 0.8797661745458701
Test F1 Score : 0.8277203654467142
